In [1]:
from google.colab import files
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SalesAnalysis").getOrCreate()

# Browse and select multiple CSV files
uploaded = files.upload()

# Load selected CSV files into PySpark DataFrames
dataframes = {}

for filename in uploaded.keys():
    df = spark.read.csv(filename, header=True, inferSchema=True)
    dataframes[filename] = df

    print(f"\nFile: {filename}")
    print("Records:", df.count())
    df.show(5)

Saving lazy_execution_stores.csv to lazy_execution_stores.csv
Saving lazy_execution_sales_25000.csv to lazy_execution_sales_25000.csv
Saving lazy_execution_products.csv to lazy_execution_products.csv

File: lazy_execution_stores.csv
Records: 8
+--------+---------+------+-----------+
|store_id|     city|region| store_type|
+--------+---------+------+-----------+
|     S01|Hyderabad| South|       Mall|
|     S02|Bengaluru| South|High Street|
|     S03|  Chennai| South|       Mall|
|     S04|   Mumbai|  West|High Street|
|     S05|     Pune|  West|       Mall|
+--------+---------+------+-----------+
only showing top 5 rows

File: lazy_execution_sales_25000.csv
Records: 25000
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+-----------

In [2]:
from pyspark.sql import functions as F
from pyspark import StorageLevel
import time

# Find the sales dataset from the uploaded files
sales_df = None

for filename, df in dataframes.items():
    if {"order_id", "quantity", "unit_price", "order_status"}.issubset(df.columns):
        sales_df = df
        print("Sales file selected:", filename)
        break

if sales_df is None:
    raise ValueError("Sales CSV not found. Please upload the sales dataset.")

print("Total Sales Records:", sales_df.count())
sales_df.show(5)

Sales file selected: lazy_execution_sales_25000.csv
Total Sales Records: 25000
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Completed|

In [3]:
# Transformation
completed_df = sales_df.filter(
F.col("order_status") == "Completed"

)

print("Transformation created")

# Action 1
print("Completed Orders:", completed_df.count())

# Action 2
completed_df.show(10)

Transformation created
Completed Orders: 21775
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Completed|       Online|
| O000006|2026-04

In [5]:
# Cache the filtered DataFrame
completed_df.cache()

print("Is Cached:", completed_df.is_cached)

# First action populates cache
completed_df.count()

# Reuse cached data
completed_df.show(10)

# Another action using cached data
completed_df.groupBy("product_id").count().show()

Is Cached: True
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Completed|       Online|
| O000006|2026-04-05|     S08|      P005|       

In [6]:
# Create filtered DataFrame
high_quantity_df = sales_df.filter(
F.col("quantity") > 5
)

# Persist using MEMORY_AND_DISK
high_quantity_df.persist(
StorageLevel.MEMORY_AND_DISK
)

# Trigger execution
high_quantity_df.count()

# Reuse persisted data
high_quantity_df.show(10)

# Check storage level
print(high_quantity_df.storageLevel)

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000006|2026-04-05|     S08|      P005|       8|      2499|           0|           UPI|   Completed|     In Store|
| O000007|2026-02-27|     S03|      P010|       8|      3999|          10|           UPI|   Completed|       Online|
| O000008|2026-03-21|     S07|      P002|       9|      2999|   

In [7]:
# Check cache status
print("Before:", completed_df.is_cached)

# Remove cache
completed_df.unpersist(blocking=True)

# Check again
print("After:", completed_df.is_cached)

# Remove the other persisted DataFrame
high_quantity_df.unpersist(blocking=True)

Before: True
After: False


DataFrame[order_id: string, order_date: date, store_id: string, product_id: string, quantity: int, unit_price: int, discount_pct: int, payment_method: string, order_status: string, sales_channel: string]

In [8]:
# Generate 40 batch IDs (0 to 39)
batch_df = spark.range(40).select(
    F.col("id").alias("batch_id")
)

batch_df.show(5)

# Combine each sales record with all 40 batch IDs
large_df = sales_df.crossJoin(batch_df)

print("Original Records:", sales_df.count())
print("Large Dataset Records:", large_df.count())

large_df.show(5)

+--------+
|batch_id|
+--------+
|       0|
|       1|
|       2|
|       3|
|       4|
+--------+
only showing top 5 rows
Original Records: 25000
Large Dataset Records: 1000000
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+--------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|batch_id|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+--------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       0|
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       1|
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       2|
| O000001|2026-06-22|     S04|      P002|       3|      2999|     

In [9]:
# Create transformations
result_df = large_df.filter(
    F.col("quantity") > 3
).withColumn(
    "revenue",
    F.col("quantity") * F.col("unit_price")
).groupBy("product_id").agg(
    F.sum("revenue").alias("total_revenue")
)

# First execution
start = time.perf_counter()

result_df.collect()

time1 = time.perf_counter() - start

# Second execution
start = time.perf_counter()

result_df.collect()

time2 = time.perf_counter() - start

print("First Execution:", round(time1, 3), "seconds")
print("Second Execution:", round(time2, 3), "seconds")

First Execution: 1.48 seconds
Second Execution: 0.105 seconds


In [10]:
# Check total records
print("Total Records:", sales_df.count())

# Check number of partitions
print("Number of Partitions:",
sales_df.rdd.getNumPartitions())

Total Records: 25000
Number of Partitions: 1


In [11]:
# Create 8 partitions
partitioned_df = sales_df.repartition(8)

print("Original Partitions:",
sales_df.rdd.getNumPartitions())

print("New Partitions:",
partitioned_df.rdd.getNumPartitions())

# Trigger execution
partitioned_df.count()

Original Partitions: 1
New Partitions: 8


25000

In [12]:
# Partition data by store_id
store_partitioned_df = sales_df.repartition(
  8,
  "store_id"
)
print("Partitions:",
store_partitioned_df.rdd.getNumPartitions())

# Inspect partition distribution
store_partitioned_df.withColumn(
"partition_id",
F.spark_partition_id()
).groupBy(
"partition_id", "store_id"
).count().orderBy(
"partition_id", "store_id"
).show(30)

Partitions: 8
+------------+--------+-----+
|partition_id|store_id|count|
+------------+--------+-----+
|           0|     S01| 3180|
|           1|     S02| 3121|
|           1|     S08| 3122|
|           3|     S06| 3033|
|           4|     S05| 3117|
|           4|     S07| 3209|
|           5|     S03| 3132|
|           5|     S04| 3086|
+------------+--------+-----+



In [14]:
# Create 8 partitions
df8 = sales_df.repartition(8)

# Reduce using repartition
df_repartition = df8.repartition(3)

# Reduce using coalesce
df_coalesce = df8.coalesce(3)

print("Repartition:",
df_repartition.rdd.getNumPartitions())

print("Coalesce:",
df_coalesce.rdd.getNumPartitions())

# Examine execution plans
print("REPARTITION PLAN")
df_repartition.explain()

print("COALESCE PLAN")
df_coalesce.explain()

Repartition: 3
Coalesce: 3
REPARTITION PLAN
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- ShuffleQueryStage 0
      +- Exchange RoundRobinPartitioning(3), REPARTITION_BY_NUM, [plan_id=1098]
         +- FileScan csv [order_id#64,order_date#65,store_id#66,product_id#67,quantity#68,unit_price#69,discount_pct#70,payment_method#71,order_status#72,sales_channel#73] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...
+- == Initial Plan ==
   Exchange RoundRobinPartitioning(3), REPARTITION_BY_NUM, [plan_id=1098]
   +- FileScan csv [order_id#64,order_date#65,store_id#66,product_id#67,quantity#68,unit_price#69,discount_pct#70,payment_method#71,order_status#72,sales_channel#73] Batched: false, DataFilters: [], Fo

**Partitioning Exercise**

In [15]:
# 1. Find the current number of partitions
print("Current Partitions:", sales_df.rdd.getNumPartitions())

# 2. Increase the number of partitions to 12
df_12 = sales_df.repartition(12)
print("Increased Partitions:", df_12.rdd.getNumPartitions())

# 3. Display the number of records in each partition
from pyspark.sql import functions as F
df_12.withColumn("partition_id", F.spark_partition_id()) \
      .groupBy("partition_id") \
      .count() \
      .orderBy("partition_id") \
      .show()

# 4. Repartition the DataFrame into 6 partitions using product_id
df_product_6 = sales_df.repartition(6, "product_id")
print("Repartitioned by product_id (Partitions):", df_product_6.rdd.getNumPartitions())

# 5. Reduce 12 partitions to 4 using coalesce()
df_coalesced_4 = df_12.coalesce(4)
print("Reduced Partitions (coalesce):", df_coalesced_4.rdd.getNumPartitions())

Current Partitions: 1
Increased Partitions: 12
+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 2084|
|           1| 2083|
|           2| 2083|
|           3| 2083|
|           4| 2083|
|           5| 2083|
|           6| 2083|
|           7| 2083|
|           8| 2083|
|           9| 2084|
|          10| 2084|
|          11| 2084|
+------------+-----+

Repartitioned by product_id (Partitions): 6
Reduced Partitions (coalesce): 4
